In [1]:
# Initialize Otter
import otter
grader = otter.Notebook("DS3000-9000_A03.ipynb")

# Assignment 3: Likelihood Functions

**DS 3000A/9000A Fall 2026**

This assignment will be graded on Gradescope.

1. Run `pip install otter-grader` first, if you don't have it installed.
2. Included in this notebook are public test cases, which you can use to verify your answers before submitting.
3. Additional private test cases will run on Gradescope upon submission. You will see your score but not the tests themselves.
4. You can revise and re-upload your submission on Gradescope up until the deadline.

In [2]:
## Install packages if neccessary.
## Uncomment and run if you need to.

# !pip install numpy scipy pandas matplotlib

In [3]:
import numpy as np
import scipy.stats as st

---

## Question 1
 
**50 Points**
 
The Gaussian (Normal) distribution is defined as:
 
$$
\mathcal{N}(x \mid \mu, \sigma^2) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\!\left(-\frac{(x - \mu)^2}{2\sigma^2}\right) .
$$
 
In the lab, $\sigma^2 = 1$ was fixed. Here, you will estimate **both** $\mu$ and $\sigma^2$ jointly from data.
 
Given i.i.d. dataset $\mathcal{D} = \{x_1, \ldots, x_N\}$, the log-likelihood is:
 
$$
\log p(\mathcal{D} \mid \mu, \sigma^2) = \sum_{n=1}^N \log \mathcal{N}(x_n \mid \mu, \sigma^2) .
$$
 
We use the following dataset of student heights (cm):

In [4]:
data = np.array([165.2, 178.4, 182.1, 159.7, 171.3, 188.6, 163.9, 175.0, 169.4, 183.2])

1. Implement `log_gaussian(x, mu, sigma)` from scratch using the logarithm of the formula above. Do **not** use `scipy.stats`. You should apply $\log$ by hand first, before putting it in code to avoid numerical problems.

2. Implement `log_likelihood(mu, sigma, data)` that computes $\log p(\mathcal{D} \mid \mu, \sigma^2)$.

3. Find the MLE estimates via **grid search** over the parameter space below. Assign the results to `mu_mle` and `sigma_mle` as Python `float`s.
   - Grid search means evaluating the log-likelihood at every point on a grid of $(\mu, \sigma)$ values and picking the best one. This is not the most efficient way to find the MLE, but it is straightforward and works well for low-dimensional problems.

In [5]:
def log_gaussian(x: np.ndarray, mu: float, sigma: float) -> np.ndarray:
    """Logarithm of the Gaussian probability density function.

    Args:
        x: Input data points, shape (N,).
        mu: Mean parameter.
        sigma: Standard deviation parameter (not variance).

    Returns:
        Log probability density at each point, shape (N,).
    """
    c = 1/np.sqrt(2*np.pi*sigma**2)
    b = -1*(x-mu)**2 / (2*sigma**2)
    return np.log(c * np.exp(b))


def log_likelihood(mu: float, sigma: float, data: np.ndarray) -> float:
    """Log-likelihood of data under a Gaussian with parameters mu and sigma.

    Args:
        mu: Mean parameter.
        sigma: Standard deviation parameter.
        data: Observed data points, shape (N,).

    Returns:
        Log-likelihood as a float.
    """
    sum = 0.0
    for x in data:
        sum += log_gaussian(x, mu, sigma)
    return sum
        
data = np.array([165.2, 178.4, 182.1, 159.7, 171.3, 188.6, 163.9, 175.0, 169.4, 183.2])

mus = np.linspace(160, 185, 500)
sigmas = np.linspace(1, 20, 500)

ll = -np.inf

for mu in mus:
    for sigma in sigmas:
        x = log_likelihood(mu, sigma, data)
        if x > ll:
            ll = x
            mu_mle = float(mu)
            sigma_mle = float(sigma)

print(f"mu_mle = {mu_mle:.2f}, sigma_mle = {sigma_mle:.2f}")

mu_mle = 173.68, sigma_mle = 8.92


In [6]:
grader.check("q1")

q1 results: All test cases passed!

---

## Question 2
 
**50 Points**
 
In lecture, it was shown that MLE under a Gaussian likelihood is equivalent to minimizing the squared error. The **Laplace distribution** plays the same role for the absolute error:
 
$$
p(x \mid \mu, s) = \frac{1}{2s} \exp\!\left(-\frac{|x - \mu|}{s}\right) .
$$
 
where $\mu \in \mathbb{R}$ is the location parameter and $s > 0$ is the scale parameter. 


Suppose we have an i.i.d. data $\mathcal{D} = \{(x_n, y_n)\}_{n=1}^N$ and have a model $f(x; w, b) = wx + b$.
We can use the Laplace distribution as our likelihood function:
 
$$
\log p(\mathcal{D} \mid w, b, s) = \sum_{n=1}^N \log p(y_n \mid f(x_n; w, b), s) .
$$
 
We generate training data as follows:

In [7]:
np.random.seed(30009000)
N = 50
x = np.random.uniform(0, 10, size=N)
TRUE_W, TRUE_B = 2.5, 1.0
y = TRUE_W * x + TRUE_B

We use this toy data so that we can verify later the learned $w$ and $b$ will be close to the true ones.

1. Implement `laplace_pdf(x, mu, s)` from scratch. Do **not** use `scipy.stats`.

2. Implement `log_likelihood_laplace(w, b, x, y)` that computes $\sum_{n=1}^N \log p(y_n \mid f(x_n; w, b), 1)$.
   - Note that we set $s = 1$ in the Laplace distribution.

4. Implement `mae(y_true, y_pred)` that computes the mean absolute error.

5. Search over the grid below and assign `w_mle`, `b_mle` (maximizing log-likelihood) and `w_mae`, `b_mae` (minimizing MAE) as Python `float`s.

In [8]:
import numpy as np
import scipy.stats as st

def laplace_pdf(x, mu, s):
    return np.log((1/(2*s)) * np.exp(-1*np.abs(x - mu) / s))

def log_likelihood_laplace(w: float, b: float, x: np.ndarray, y: np.ndarray) -> float:
    """Log-likelihood under a linear model with Laplace noise (scale=1).

    Args:
        w: Weight parameter.
        b: Bias parameter.
        x: Input array, shape (N,).
        y: Target array, shape (N,).

    Returns:
        Log-likelihood as a float.
    """
    
    y_pred = w*x + b
    sum = 0.0
    for i in range(y.shape[0]):
        sum += laplace_pdf(y[i], y_pred[i], 1)
    return sum

def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Mean absolute error.

    Args:
        y_true: True values, shape (N,).
        y_pred: Predicted values, shape (N,).

    Returns:
        MAE as a float.
    """
    return np.mean(np.abs(y_true - y_pred))


# Possible values for w and b
ws = np.linspace(1.5, 3.5, 200)
bs = np.linspace(0.0, 2.0, 200)


max_ll = -np.inf
min_mae = np.inf
for w in ws:
    for b in bs:
        y_pred = w*x+b
        ll = log_likelihood_laplace(w, b, x, y)
        mean_abs_err = mae(y, y_pred)
        
        if max_ll < ll:
            w_mle = w
            b_mle = b
            max_ll = ll
            
        if min_mae > mean_abs_err:
            w_mae = w
            b_mae = b
            min_mae = mean_abs_err

print(f"MLE: w = {w_mle:.3f}, b = {b_mle:.3f}")
print(f"MAE: w = {w_mae:.3f}, b = {b_mae:.3f}")

MLE: w = 2.495, b = 1.025
MAE: w = 2.495, b = 1.025


In [9]:
grader.check("q2")

q2 results: All test cases passed!

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [10]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False, run_tests=True)

Running your submission against local test cases...


Your submission received the following results when run against available test cases:

    q1 results: All test cases passed!

    q2 results: All test cases passed!
